In [1]:
import numpy as np

dataHandler =  open("Project_data.xls", "r")
header = dataHandler.readline().strip().split(",")  
data = np.loadtxt(dataHandler, delimiter=",")        

X = data[:, :5]  
y = data[:, 5]    


import numpy as np

def add_bias_column(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    m = X.shape[0]
    return np.hstack((np.ones((m, 1)), X))

def standardize_features(X: np.ndarray):
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    sigma = np.where(sigma == 0, 1.0, sigma) 
    Xs = (X - mu) / sigma
    return Xs, mu, sigma

def compute_cost(Xb: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    y = np.asarray(y, dtype=float).reshape(-1)
    r = Xb @ w - y
    m = y.size
    return float((r @ r) / (2.0 * m))

def compute_grad(Xb: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=float).reshape(-1)
    r = Xb @ w - y
    m = y.size
    return (Xb.T @ r) / m
def gradient_descent(
    X: np.ndarray,
    y: np.ndarray,
    alpha: float = 0.01,
    max_iter: int = 50_000,
    tol_cost: float = 1e-10,
    tol_update: float = 1e-10,
    standardize: bool = True,
    store_w: bool = True,
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)

    if standardize:
        Xs, mu, sigma = standardize_features(X)
    else:
        Xs, mu, sigma = X, None, None

    Xb = add_bias_column(Xs)
    w = np.zeros(Xb.shape[1], dtype=float)

    history = []
    w_hist = [] 

    prev_cost = compute_cost(Xb, y, w)
    history.append(prev_cost)
    if store_w:
        w_hist.append(w.copy())

    iters_used = 0
    for it in range(1, max_iter + 1):
        iters_used = it

        grad = compute_grad(Xb, y, w)
        w_new = w - alpha * grad

        new_cost = compute_cost(Xb, y, w_new)
        history.append(new_cost)
        if store_w:
            w_hist.append(w_new.copy())

        if not np.isfinite(new_cost):
            raise RuntimeError("Diverged (non-finite cost). Try smaller alpha.")
        
        if new_cost > prev_cost * 1.001:
            raise RuntimeError("Cost increased — try smaller alpha.")
        if abs(new_cost - prev_cost) < tol_cost or np.linalg.norm(w_new - w) < tol_update:
            w = w_new
            break

        w = w_new
        prev_cost = new_cost

    if store_w:
        w_hist = np.vstack(w_hist) 

    return w, history, mu, sigma, iters_used, Xb, w_hist

w, history, mu, sigma, iters_used, Xb, w_hist = gradient_descent(X, y)


def lin_regression_std(w, X, index, mu, sigma):
    x = (X[index] - mu) / sigma
    return float(x @ w[1:] + w[0])

index = 1422
estimate = lin_regression_std(w, X, index, mu, sigma)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

w = [364.54435751864537, 2.7268522381452276, 16.68207764207219, 8.930510176877332, -4.91824389113619, -14.888434270235448]
mu = [594.0162443144899, 3.247568810916178, 4.30462508619557, 893.7914230019493, 48.512020792722545]
sigma = [15.915684963900695, 0.4786019232456688, 0.33270119546324906, 567.0908296976912, 5.339696959818941]

feature_names = [
    "Initial Structural Strength (MPa)",
    "Thickness (mm)",
    "Speed (m/s)",
    "Cooling Fans Power (W)",
    "Number of cooling fans",
]
target_name = "Final Structural Strength (MPa)"

class PredictorUI:
    def __init__(self, w, X, mu, sigma, feature_names, target_name):
        self.w, self.X = np.asarray(w, float), np.asarray(X, float)
        self.mu = np.zeros(len(feature_names)) if mu is None else np.asarray(mu, float)
        self.sigma = np.ones(len(feature_names)) if sigma is None else np.asarray(sigma, float)
        self.feature_names, self.target_name = list(feature_names), target_name
        self.n = len(self.feature_names)
        self.saved_rows, self._syncing = [], False
        self._changing_feature = False

        self.inputs = [widgets.IntText if i==self.n-1 else widgets.FloatText for i in range(self.n)]
        self.inputs = [self.inputs[i](description=self.feature_names[i],
                                      value=int(round(self.mu[i])) if i==self.n-1 else float(self.mu[i]),
                                      layout=widgets.Layout(width="300px"))
                       for i in range(self.n)]

        self.vary = widgets.Dropdown(options=[(n,i) for i,n in enumerate(self.feature_names)],
                                     value=0, description="Vary:", layout=widgets.Layout(width="500px"))
        self.slider = widgets.FloatSlider(description="Value:", continuous_update=True,
                                          layout=widgets.Layout(width="500px"))
        self._ignore_slider_events = 0
        self.out_plot, self.out_saved = widgets.Output(), widgets.Output()

        self.ranges = [self._range_for(i) for i in range(self.n)]
        self.ymin, self.ymax = self._fixed_ylim()

        self._wire()
        self._configure_slider(self.vary.value)
        self._sync_slider_from_input()
        self.redraw()

        self.ui = widgets.VBox([
            widgets.HTML("<b>Add your values here</b>"),
            widgets.HBox([widgets.VBox(self.inputs[:3]), widgets.VBox(self.inputs[3:])]),
            widgets.HBox([self.vary, self.slider]),
            self.out_plot,
            self.out_saved
        ])

    def show(self): display(self.ui)

    def predict(self, x_raw):
        x = np.asarray(x_raw, float)
        xs = (x - self.mu) / self.sigma
        return float(self.w[0] + xs @ self.w[1:])

    def _current_x(self):
        x = [float(int(w.value)) if i==self.n-1 else float(w.value) for i,w in enumerate(self.inputs)]
        x[-1] = int(round(x[-1]))
        return x

    def _range_for(self, i):
        lo, hi = float(np.nanmin(self.X[:, i])), float(np.nanmax(self.X[:, i]))
        if (not np.isfinite(lo)) or (not np.isfinite(hi)) or (lo == hi):
            lo, hi = float(self.mu[i] - 1.0), float(self.mu[i] + 1.0)
        return (hi, lo) if lo > hi else (lo, hi)

    def _fixed_ylim(self):
        Xs = (self.X - self.mu) / self.sigma
        y = np.hstack([np.ones((Xs.shape[0], 1)), Xs]) @ self.w
        ymin, ymax = float(np.nanmin(y)), float(np.nanmax(y))
        pad = 0.05 * (ymax - ymin if ymax > ymin else 1.0)
        return ymin - pad, ymax + pad

    def _set_slider_safely(self, *, lo, hi, step, fmt, value):
        self._ignore_slider_events = 3
    
        self.slider.unobserve(self._on_slider, names="value")
        self._syncing = True
        try:
            with self.slider.hold_trait_notifications():
                self.slider.min = lo
                self.slider.max = hi
                self.slider.step = step
                self.slider.readout_format = fmt
                self.slider.value = value
        finally:
            self._syncing = False
            self.slider.observe(self._on_slider, names="value")

    def _configure_slider(self, i):
        lo, hi = self.ranges[i]
    
        if i == self.n - 1:  
            step, fmt = 1.0, ".0f"
            desired = float(int(self.inputs[i].value))
        else:
            step, fmt = max((hi - lo) / 200.0, 0.01), ".2f"
            desired = float(self.inputs[i].value)
    
        if desired < lo:
            lo = desired
        if desired > hi:
            hi = desired
    
        pad = 0.02 * (hi - lo if hi > lo else 1.0)
        lo -= pad
        hi += pad
    
        self._set_slider_safely(lo=lo, hi=hi, step=step, fmt=fmt, value=desired)

    def _sync_slider_from_input(self):
        if self._syncing:
            return
    
        i = self.vary.value
        v = float(int(self.inputs[i].value)) if i == self.n - 1 else float(self.inputs[i].value)
    
        lo, hi = self.slider.min, self.slider.max
    
        if v < lo:
            lo = v
        if v > hi:
            hi = v
    
        self._set_slider_safely(lo=lo, hi=hi, step=self.slider.step, fmt=self.slider.readout_format, value=v)

    def _sync_input_from_slider(self):
        if self._syncing: return
        i, v = self.vary.value, float(self.slider.value)
        self._syncing = True
        try:
            with self.inputs[i].hold_trait_notifications():
                self.inputs[i].value = int(round(v)) if i==self.n-1 else float(v)
        finally:
            self._syncing = False

    def redraw(self, *_):
        i = self.vary.value
        x = self._current_x()
        yhat = self.predict(x)
        with self.out_plot:
            self.out_plot.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(6.8, 4.2))
            ax.plot([x[i]], [yhat], marker="x", linestyle="None")
            ax.set_xlabel(self.feature_names[i])
            ax.set_ylabel(self.target_name)
            ax.set_title("Live prediction (with a point)")
            ax.set_xlim(self.slider.min, self.slider.max)
            ax.set_ylim(self.ymin, self.ymax)
            ax.grid(True)
            plt.show()
            print(f"{self.target_name} = {yhat:.2f} MPa")

    def _wire(self):
        self.vary.observe(self._on_vary, names="value")
        self.slider.observe(self._on_slider, names="value")
        for idx, w in enumerate(self.inputs):
            w.observe(lambda ch, i=idx: self._on_input(ch, i), names="value")
  
    def _on_vary(self, change):
        if self._syncing:
            return
    
        self._changing_feature = True
        try:
            self._configure_slider(change["new"])
            self._sync_slider_from_input()
        finally:
            self._changing_feature = False
    
        self.redraw()

    def _on_slider(self, change):
        if self._ignore_slider_events > 0:
            self._ignore_slider_events -= 1
            return
    
        if self._syncing or self._changing_feature:
            return
    
        self._sync_input_from_slider()
        self.redraw()
    
    def _on_input(self, change, idx):
        if idx == self.vary.value:
            self._sync_slider_from_input()
        self.redraw()

ui = PredictorUI(w=w, X=X, mu=mu, sigma=sigma,
                feature_names=feature_names, target_name=target_name)
ui.show()